In [0]:
#  Install Great Expectations (Run once)
%pip install great-expectations==0.17.23
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import great_expectations as ge
from pyspark.sql.functions import col, count as sql_count

In [0]:
# Read Data from Bronze Delta Lake
airports_bronze_path = "s3://travel-analytics-bronze/delta/bronze/airports/"
df_airports = spark.read.format("delta").load(airports_bronze_path)

print("=" * 80)
print("AIRPORTS VALIDATION WITH GREAT EXPECTATIONS")
print("=" * 80)
print(f"Total records: {df_airports.count()}")
print("\n--- Schema ---")
df_airports.printSchema()


AIRPORTS VALIDATION WITH GREAT EXPECTATIONS
Total records: 105

--- Schema ---
root
 |-- _airbyte_ab_id: string (nullable = true)
 |-- _airbyte_emitted_at: timestamp (nullable = true)
 |-- city: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- airport_id: long (nullable = true)
 |-- _ab_cdc_lsn: double (nullable = true)
 |-- airport_name: string (nullable = true)
 |-- airport_type: string (nullable = true)
 |-- _ab_cdc_deleted_at: string (nullable = true)
 |-- _ab_cdc_updated_at: string (nullable = true)
 |-- _airbyte_additional_properties: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)



In [0]:
ge_df = ge.from_pandas(df_airports.toPandas())

print("\nRunning Great Expectations validation...")


Running Great Expectations validation...


In [0]:
# Run expectations
result1 = ge_df.expect_column_values_to_not_be_null(column="airport_id")
result2 = ge_df.expect_column_values_to_be_unique(column="airport_id")
result3 = ge_df.expect_column_values_to_not_be_null(column="city")
result4 = ge_df.expect_column_values_to_not_be_null(column="airport_name")

print(f"✓ airport_id NOT NULL: {result1['success']}")
print(f"✓ airport_id UNIQUE: {result2['success']}")
print(f"✓ city NOT NULL: {result3['success']}")
print(f"✓ airport_name NOT NULL: {result4['success']}")

✓ airport_id NOT NULL: True
✓ airport_id UNIQUE: True
✓ city NOT NULL: True
✓ airport_name NOT NULL: True


In [0]:
print("\n--- Applying Validations ---")

# Start with all data
df_valid = df_airports
df_invalid_list = []

# Rule 1: airport_id NOT NULL
df_invalid_null_id = df_valid.filter(col("airport_id").isNull())
if df_invalid_null_id.count() > 0:
    df_invalid_list.append(df_invalid_null_id)
    print(f"  → Found {df_invalid_null_id.count()} rows with airport_id = NULL")
df_valid = df_valid.filter(col("airport_id").isNotNull())

# Rule 2: airport_id UNIQUE
duplicates = df_valid.groupBy("airport_id").agg(sql_count("*").alias("cnt")).filter(col("cnt") > 1)
duplicate_ids = [row.airport_id for row in duplicates.collect()]
if duplicate_ids:
    df_invalid_dup = df_valid.filter(col("airport_id").isin(duplicate_ids))
    df_invalid_list.append(df_invalid_dup)
    print(f"  → Found {df_invalid_dup.count()} rows with duplicate airport_id")
    df_valid = df_valid.filter(~col("airport_id").isin(duplicate_ids))

# Rule 3: airport_name NOT NULL
df_invalid_name = df_valid.filter(col("airport_name").isNull())
if df_invalid_name.count() > 0:
    df_invalid_list.append(df_invalid_name)
    print(f"  → Found {df_invalid_name.count()} rows with airport_name = NULL")
df_valid = df_valid.filter(col("airport_name").isNotNull())

# Rule 4: city NOT NULL
df_invalid_city = df_valid.filter(col("city").isNull())
if df_invalid_city.count() > 0:
    df_invalid_list.append(df_invalid_city)
    print(f"  → Found {df_invalid_city.count()} rows with city = NULL")
df_valid = df_valid.filter(col("city").isNotNull())



--- Applying Validations ---


In [0]:
# Combine Invalid Records

if df_invalid_list:
    df_invalid = df_invalid_list[0]
    for df_temp in df_invalid_list[1:]:
        df_invalid = df_invalid.union(df_temp)
    df_invalid = df_invalid.distinct()
else:
    df_invalid = spark.createDataFrame([], df_airports.schema)

valid_count = df_valid.count()
invalid_count = df_invalid.count()
total_count = df_airports.count()

print("\n" + "=" * 80)
print("VALIDATION RESULTS")
print("=" * 80)
print(f"✅ Valid records:   {valid_count} ({valid_count/total_count*100:.2f}%)")
print(f"❌ Invalid records: {invalid_count} ({invalid_count/total_count*100:.2f}%)")



VALIDATION RESULTS
✅ Valid records:   105 (100.00%)
❌ Invalid records: 0 (0.00%)


In [0]:
# Write Invalid Records to Quarantine

if invalid_count > 0:
    quarantine_path = "s3://travel-analytics-bronze/Quarantine/Airports"
    
    df_invalid.write \
        .format("parquet") \
        .mode("append") \
        .save(quarantine_path)
    
    print(f"\n❌ Invalid records sent to Quarantine: {quarantine_path}")
    print("\n--- Sample Invalid Records ---")
    df_invalid.show(10, truncate=False)
else:
    print(f"\n✅ All {valid_count} records passed validation!")

print("\n" + "=" * 80)
print("✅ VALIDATION COMPLETED!")
print("=" * 80)
print(f"Valid records remain in Bronze: {airports_bronze_path}")
if invalid_count > 0:
    print(f"Invalid records in Quarantine: s3://travel-analytics-bronze/Quarantine/Airports")


✅ All 105 records passed validation!

✅ VALIDATION COMPLETED!
Valid records remain in Bronze: s3://travel-analytics-bronze/delta/bronze/airports/
